# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a comprehensive guide to loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library. All entities (record sets, fields, etc.) are referenced by their Croissant schema `@id` fields as required for reproducibility.

### Dataset Source
The dataset source is provided as a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed. If running on Colab or a clean environment, un-comment the following line:
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
# dataset.metadata is an object providing dataset-level information
print(f"Dataset name: {dataset.metadata.name}")
print(f"Dataset description: {dataset.metadata.description}")
print(f"Dataset identifier: {dataset.metadata.identifier}")
print(f"Published on: {dataset.metadata.datePublished}")
print(f"Fields: {dataset.metadata.keywords}")

## 2. Data Overview

Review available record sets, the fields within each record set, and their Croissant schema IDs. This overview will guide you to reference all data by schema `@id`.

**Note:** We enumerate the record sets, then for each, enumerate the fields and columns, all by their `@id` field.

In [ ]:
# Inspect available record sets and fields
record_sets = dataset.metadata.recordSets
print(f"Number of record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record Set: {rs['@id']}")
    name = rs.get('name', rs['@id'])
    print(f"  Name: {name}")
    print("  Fields:")
    for f in rs['fields']:
        field_id = f['@id']
        field_name = f.get('name', field_id)
        dtype = f.get('dataType', 'Unknown')
        print(f"    - {field_id} (name: {field_name}, type: {dtype})")
    print("  Columns:")
    for c in rs['columns']:
        col_id = c['@id']
        col_name = c.get('name', col_id)
        print(f"    - {col_id} (name: {col_name})")
    print()

## 3. Data Extraction

Load data from the primary record set(s) into a DataFrame for analysis. All references are via `@id`.

In [ ]:
# List all record set @id's
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSets]
print("Record sets in this dataset (by @id):", record_set_ids)

# We will extract the main one -- typically @id ends with 'recordset-1'
# If unsure, print names from previous cell and pick the main data table.

# For demonstration, we'll use the first record set available
target_record_set_id = record_set_ids[0]

# Load records from the chosen record set into a DataFrame
records = list(dataset.records(record_set=target_record_set_id))
df = pd.DataFrame(records)

print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns from record set {target_record_set_id}")
print("Available columns (field @id):")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)

Let's process the data. You can filter records, normalize numeric fields, and group/categorize data, referencing fields by their `@id`.

In [ ]:
# For this example, pick a numeric field (e.g., coverage percentage or molecular weight)
# Inspect first for numeric-like columns
numeric_field_candidates = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
if not numeric_field_candidates:
    # Try to auto-infer numeric columns by attempting conversion
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            pass
    numeric_field_candidates = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]

print("Numeric fields detected (@id):", numeric_field_candidates)

# Select a numeric field for analysis
if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]  # Pick the first one
else:
    raise RuntimeError('No numeric fields found to analyze.')

threshold = df[numeric_field].mean()  # Use mean value for thresholding
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a non-numeric field
group_field_candidates = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field]
if group_field_candidates:
    group_field = group_field_candidates[0]
    print(f"Grouping by field (@id): {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped data by {group_field} (mean {numeric_field}):")
    print(grouped_df.head())
else:
    print('No suitable string group fields found.')

## 5. Visualization

Visualize a distribution of the numeric field across all records in the main record set, and show the result of the grouping as a bar plot if a group field was found.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field} (by @id)")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If data was grouped before
if 'grouped_df' in locals():
    plt.figure(figsize=(10, 4))
    grouped_df = grouped_df.sort_values(ascending=False)[:20]  # Show top 20
    grouped_df.plot(kind='bar')
    plt.title(f"Mean {numeric_field} by {group_field} (top 20)")
    plt.ylabel(f"Mean {numeric_field}")
    plt.xlabel(group_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

We have:
* Loaded the FAIR² dataset from a Croissant schema URL via `mlcroissant`
* Inspected and listed available record sets and fields by their schema `@id`
* Loaded records from the main record set, extracted fields, and performed data analysis
* Applied basic EDA: numeric filtering, normalization, and grouping
* Visualized distributions and grouped summaries

**All references to dataset structure and fields use the Croissant schema `@id` for clarity and reproducibility.**

For further analysis, you can now proceed with domain-specific modeling or advanced visualizations!